<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/line_dummy_image_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LINE ダミー画像 Bot（Colab動作確認版）

LINE公式アカウントに送ったテキストを、そのまま**画像に描いて**返します。
Gemini等の外部APIは使わず、PILで画像を生成するので**課金不要**。
署名検証 → 画像生成 → ngrok公開URLで画像返信、という配管が正しく動くかの確認用です。

**仕組み（前の構成と同じ）**
- LINEテキスト受信 → 画像生成（今回はダミー） → 一時保管 → ngrok経由の公開URLで画像メッセージを返信
- 画像生成は別スレッドで行い、Webhookには即200を返す
- LINE画像メッセージはJPEG必須なのでJPEGで配信

あとでGeminiに戻すときは、`generate_image_jpeg()` の中身を画像生成APIの呼び出しに置き換えるだけです（Flask以降は無修正）。

セルは上から順に実行してください。

## 1. ライブラリと日本語フォントのインストール

日本語が□（豆腐）にならないよう Noto CJK フォントを入れます。

In [ ]:
!pip install flask pyngrok pillow --quiet
!apt-get -qq install -y fonts-noto-cjk > /dev/null
print("done")

done


## 2. 設定値の入力

- `CHANNEL_SECRET` / `CHANNEL_ACCESS_TOKEN` … LINE Developers の値
- `NGROK_AUTHTOKEN` … https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
CHANNEL_SECRET = "e3a0d7a0e143575da45422bcb27ddb62" #"ここにChannel secret" #
CHANNEL_ACCESS_TOKEN = "7cfnmgNUvcDbxsjh+PGZOODJf5CoEcRegNhgfVMpjU0xab/vBNQiZe97tXQjhgDqUzypDAale03H39tSk7o+fRJpB1kvCjwdqU3HHiyMsmDf+C7OaRrvbffQomf8I5bHlgc2yKDOna0zBEgdzw4ItAdB04t89/1O/w1cDnyilFU=" #"ここにChannel access token" #
NGROK_AUTHTOKEN = "2HhWtFPzcpgWIRTD67dJ7y59sYx_3nvbFfQgmH2JPJ7Aevg4t" #"ここにngrokのauthtoken"

LINE_REPLY_URL = "https://api.line.me/v2/bot/message/reply"
print("設定OK")

設定OK


## 3. ダミー画像の生成関数

受け取ったテキストを背景色つきの画像に描いて、**JPEGバイト列**を返します。
インターフェイス（関数名・戻り値）は前の構成と同じなので、後でGemini呼び出しに差し替え可能です。

In [ ]:
import os, hashlib, datetime
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont

# 日本語が描けるフォントを探す（無ければ既定フォント＝日本語は表示できない）
_FONT_CANDIDATES = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    "/usr/share/fonts/opentype/noto/NotoSansCJKjp-Regular.otf",
    "/usr/share/fonts/truetype/fonts-japanese-gothic.ttf",
]
def _find_font_path():
    for p in _FONT_CANDIDATES:
        if os.path.exists(p):
            return p
    return None
_FONT_PATH = _find_font_path()
print("使用フォント:", _FONT_PATH or "(既定・日本語非対応)")

def _load_font(size):
    if _FONT_PATH:
        try:
            return ImageFont.truetype(_FONT_PATH, size)
        except Exception:
            pass
    return ImageFont.load_default()

def _bg_color(text):
    """テキストごとに色を変える（暗めの色＝白文字が映える）。"""
    h = hashlib.md5(text.encode("utf-8")).digest()
    return (40 + h[0] % 120, 40 + h[1] % 120, 40 + h[2] % 120)

def _wrap(draw, text, font, max_width):
    """日本語向けに幅で折り返し（スペースに依存しない）。"""
    lines, cur = [], ""
    for ch in text:
        if ch == "\n":
            lines.append(cur); cur = ""; continue
        if draw.textlength(cur + ch, font=font) <= max_width:
            cur += ch
        else:
            lines.append(cur); cur = ch
    lines.append(cur)
    return lines

def generate_image_jpeg(prompt_text):
    """受け取ったテキストを描いたダミー画像のJPEGバイト列を返す。"""
    W = H = 1024
    img = Image.new("RGB", (W, H), _bg_color(prompt_text))
    draw = ImageDraw.Draw(img)
    margin = 80

    # 上部ラベル
    head_font = _load_font(40)
    draw.text((margin, 50), "DUMMY IMAGE", font=head_font, fill=(255, 255, 255))
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    small = _load_font(28)
    draw.text((margin, 110), f"受信: {ts}", font=small, fill=(220, 220, 220))

    # 本文（受け取ったテキスト）を中央に折り返し描画
    body_font = _load_font(64)
    body = prompt_text if prompt_text else "(空のメッセージ)"
    lines = _wrap(draw, body, body_font, W - margin * 2)
    line_h = body_font.size + 18
    total_h = line_h * len(lines)
    y = (H - total_h) // 2
    for ln in lines:
        w = draw.textlength(ln, font=body_font)
        draw.text(((W - w) // 2, y), ln, font=body_font, fill=(255, 255, 255))
        y += line_h

    buf = BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return buf.getvalue()

# 動作確認: 1枚作ってバイト数を表示
_t = generate_image_jpeg("テスト：これはダミー画像です")
print("生成テスト OK / bytes:", len(_t))

使用フォント: /usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc
生成テスト OK / bytes: 40521


## 4. 署名検証・返信・画像保管の関数

（前の構成と同一）署名検証はHMAC-SHA256。生成画像はメモリに一時保管し `/images/<id>.jpg` で配信します。

In [ ]:
import hmac, hashlib, base64, json, uuid, time
import urllib.request

IMAGE_STORE = {}   # {id: (jpeg_bytes, 作成時刻)}
STORE_TTL = 600    # 10分で自動掃除

def verify_signature(body_bytes, signature):
    if not CHANNEL_SECRET or CHANNEL_SECRET == "ここにChannel secret":
        return True  # 未設定時はスキップ(動作確認用)
    digest = hmac.new(CHANNEL_SECRET.encode("utf-8"), body_bytes, hashlib.sha256).digest()
    expected = base64.b64encode(digest).decode("utf-8")
    return hmac.compare_digest(expected, signature)

def put_image(jpeg_bytes):
    img_id = uuid.uuid4().hex
    IMAGE_STORE[img_id] = (jpeg_bytes, time.time())
    now = time.time()
    for k in list(IMAGE_STORE):
        if now - IMAGE_STORE[k][1] > STORE_TTL:
            IMAGE_STORE.pop(k, None)
    return img_id

def _post_line_reply(data):
    req = urllib.request.Request(
        LINE_REPLY_URL,
        data=json.dumps(data).encode("utf-8"),
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {CHANNEL_ACCESS_TOKEN}"},
        method="POST",
    )
    urllib.request.urlopen(req)

def reply_text(reply_token, text):
    _post_line_reply({"replyToken": reply_token,
                      "messages": [{"type": "text", "text": text}]})

def reply_image(reply_token, image_url):
    _post_line_reply({"replyToken": reply_token,
                      "messages": [{"type": "image",
                                    "originalContentUrl": image_url,
                                    "previewImageUrl": image_url}]})
print("関数を定義しました")

関数を定義しました


## 5. Flaskアプリの定義

（前の構成と同一）`/callback` は画像生成を別スレッドに投げて即200を返し、`/images/<id>.jpg` で配信します。

In [ ]:
from flask import Flask, request, Response
import threading

app = Flask(__name__)
PUBLIC_URL = None  # ngrok起動後にセット

def handle_event_async(reply_token, user_text):
    try:
        jpeg = generate_image_jpeg(user_text)
        if jpeg is None:
            reply_text(reply_token, "画像を生成できませんでした。")
            return
        img_id = put_image(jpeg)
        image_url = f"{PUBLIC_URL}/images/{img_id}.jpg"
        reply_image(reply_token, image_url)
    except Exception as e:
        print("生成/返信エラー:", e)
        try:
            reply_text(reply_token, f"エラー: {e}")
        except Exception:
            pass

@app.route("/callback", methods=["POST"])
def callback():
    body_bytes = request.get_data()
    signature = request.headers.get("X-Line-Signature", "")
    if not verify_signature(body_bytes, signature):
        return "NG", 400
    data = json.loads(body_bytes.decode("utf-8"))
    for event in data.get("events", []):
        if event.get("type") != "message":
            continue
        message = event.get("message", {})
        if message.get("type") != "text":
            continue
        reply_token = event.get("replyToken")
        user_text = message.get("text", "")
        threading.Thread(target=handle_event_async,
                         args=(reply_token, user_text), daemon=True).start()
    return "OK", 200

@app.route("/images/<img_id>.jpg", methods=["GET"])
def serve_image(img_id):
    item = IMAGE_STORE.get(img_id)
    if item is None:
        return "Not found", 404
    return Response(item[0], mimetype="image/jpeg")

@app.route("/", methods=["GET"])
def index():
    return "LINE dummy image bot is running."
print("Flaskアプリを定義しました")

Flaskアプリを定義しました


## 6. ngrokでトンネルを開いてサーバー起動

表示される `Webhook URL` を LINE Developers の Webhook URL に貼り、「検証」を押してください。
このセルは起動したままになります。

In [ ]:
from pyngrok import ngrok, conf

# 1. 何よりも先にauthtokenを設定する
#    （これより前にngrokを起動するとERR_NGROK_4018になる）
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 2. 認証後に既存トンネルを掃除
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

PORT = 5000
PUBLIC_URL = ngrok.connect(PORT).public_url
print("=" * 55)
print(f"Webhook URL: {PUBLIC_URL}/callback")
print("↑ これを LINE Developers の Webhook URL に設定")
print("=" * 55)

import threading
def run():
    app.run(port=PORT, use_reloader=False)
threading.Thread(target=run, daemon=True).start()
print("サーバー起動中。LINEにテキストを送ると、その文字を描いた画像が返ります。")

Webhook URL: https://c383-8-228-108-136.ngrok-free.app/callback
↑ これを LINE Developers の Webhook URL に設定
サーバー起動中。LINEにテキストを送ると、その文字を描いた画像が返ります。


## 補足

- まず `https://xxxx.ngrok-free.app/images/...` をブラウザで直接開けるか確認すると切り分けが楽です（テキストは返るのに画像が出ない場合、ngrok無料版の警告ページが原因のことが多い）。
- ここが通れば配管は正常。あとは `generate_image_jpeg()` の中身を Gemini 等の画像生成APIに差し替えるだけで本番化できます（要・課金有効化）。